## 💡 IDEA DEL PROYECTO

Al principio no tenía claro sobre qué hacer el EDA. Mi primera idea fue encontrar una base de datos de jugadores de fútbol de la liga española, agruparlos por características físicas y ver si la relación entre altura y peso guardaba relación con el tipo de lesiones que podían padecer. Sin embargo, los datos que encontré por internet no me convencieron.

Decidí entonces tirar por los datos meteorológicos, ya que sabía que existían múltiples formas de obtener este tipo de información de forma libre y gratuita. La premisa principal era **discernir si la temperatura del Mar Mediterráneo frente a la costa de Valencia tenía efectos directos en la aparición de las DANAs** y si estos fenómenos eran cada vez más frecuentes.

## 🛠️ CREACIÓN Y ORGANIZACIÓN DEL PROYECTO

Lo primero que hice fue crear el repositorio en GitHub y clonármelo al ordenador. A continuación, planifiqué junto con la IA la estructura de carpetas y el calendario de trabajo hasta la fecha de entrega. Esta estructura inicial se ha mantenido prácticamente intacta hasta el final:

```text
mi_proyecto_eda_valencia/
│
├── data/
│   ├── raw/                 # Datos originales descargados de las APIs (sin modificar)
│   │   ├── aemet_valencia_2021_2026.json
│   │   └── open_meteo_mar_2021_2026.csv
│   └── processed/           # Datos limpios, unidos y listos para analizar
│       └── df_valencia_mar_unificado.csv
│
├── notebooks/
│   └── eda_valencia_clima_mar.ipynb  # El Jupyter Notebook principal
│
├── src/                     # Código en Python modular (para no saturar el notebook)
│   ├── __init__.py
│   ├── api_aemet.py         # Funciones para conectar y descargar de AEMET
│   ├── api_open_meteo.py    # Funciones para conectar y descargar de Open-Meteo
│   └── preprocessing.py     # Funciones de limpieza y unificación
│
├── tests/
│   └── test_api.py          # Script de prueba para validar conexiones
│
├── .env                     # Credenciales secretas (API Key de AEMET)
├── .gitignore               # Para ignorar .env, datos pesados y checkpoints
├── requirements.txt         # Librerías necesarias (pandas, requests, seaborn, etc.)
└── README.md                # Explicación del proyecto y guía de ejecución
```

Después de esto, configuré el entorno virtual (`.venv`), creé el archivo `requirements.txt` e instalé los paquetes necesarios (un proceso que fuimos alimentando durante todo el desarrollo). Con todo listo, pude dar comienzo al análisis.

## 📡 CONEXIÓN Y EXTRACCIÓN: AEMET OPEN DATA

No me veía con capacidad para enfrentarme al *web scraping* con los plazos que manejábamos, y descargar un dataset genérico de Kaggle me parecía demasiado sencillo. Así que opté por el camino del medio: hacer peticiones directas a la **API de Aemet Open Data** a través de Python.

Antes de montar el script definitivo, tuve que hacer un par de tests para comprender cómo funcionaban los endpoints y las claves de acceso, algo que me sirvió para interiorizar el flujo de trabajo de una API REST de forma clara.

![test_api.png](./img/test_api.png)

## 🌊 LOS DATOS MARINOS: COPERNICUS

Conseguir los registros sobre la temperatura del Mar Mediterráneo no fue tan trivial. Tras buscar en varios portales, descubrí el **Servicio de Vigilancia Medioambiental Marina de Copernicus (CMEMS)**. 

Este servicio de observación de la Tierra cuenta con mapas interactivos increíbles, pero extraer la información programáticamente requiere lidiar con formatos complejos. Tras conseguir descargar la información de la temperatura superficial del mar, me topé con un archivo en formato NetCDF (`.nc`). Tuve que tirar mano de la IA y de librerías específicas para transformarlo a una estructura tabular que mi entorno de trabajo pudiera leer sin problemas.

![Copernicus.png](./img/Copernicus.png)

## 🧹 PRIMEROS PASOS Y LIMPIEZA INICIAL

Una vez descargados ambos archivos, los convertí en DataFrames de Pandas para inspeccionarlos, detectar valores nulos y entender los tipos de datos. 

* Cambié los formatos de fecha de strings a objetos `datetime`.
* Limpié los campos numéricos.
* Detecté 5 valores nulos en columnas de precipitación, los cuales decidí imputar utilizando la media de registros con condiciones similares.
* Por último, integré ambos DataFrames en una única tabla unificada, guardándola como `.csv` en la carpeta `data/processed/` para agilizar las iteraciones posteriores.

![Nulos1.png](./img/Nulos1.png)

![Limpieza_Nulos.png](./img/Limpieza_Nulos.png)

## 📈 AMPLIACIÓN DEL ALCANCE TEMPORAL

Tras un primer ciclo de análisis exploratorio donde conseguí visualizar las primeras tablas, tomé la decisión estratégica de **aumentar el rango temporal**. Lo que en un principio iban a ser 5 años terminaron convirtiéndose en **35 años de serie histórica**. 

Esta ampliación abrió la puerta a incorporar nuevas variables atmosféricas clave, lo que exigió una segunda ronda de incorporación y depuración de datos.

![graficas_Enero.jpg](./img/graficas_Enero.jpg)

## 🔄 SEGUNDA FASE DE LIMPIEZA: OPEN-METEO

Las nuevas variables añadidas (presión atmosférica, humedad relativa, velocidad del viento...) aportaban una riqueza analítica enorme, pero venían con un hándicap: aproximadamente la mitad de los registros históricos presentaban valores nulos (`NaN`). 

Tras valorar opciones, decidí buscar fuentes complementarias y encontré la API de **Open-Meteo**, desde donde pude descargar un archivo CSV completo con los datos del periodo completo. Tras un proceso de saneamiento, homologación de unidades y fusión, conseguí consolidar el DataFrame definitivo.

![Nulos2.png](./img/Nulos2.png)

## 🧮 ANÁLISIS DE CORRELACIÓN (PEARSON Y SPEARMAN)

Con el dataset definitivo listo, calculé las matrices de correlación de Pearson y Spearman para todo el conjunto de datos y detectar asociaciones lineales y monótonas.

Para comprobar si el ciclo anual estacional distorsionaba la señal, repetí la operación **aislando únicamente los meses de agosto a diciembre** (el trimestre crítico de las DANAs y el otoño meteorológico). Aunque el comportamiento general se mantuvo coherente, este filtro permitió afinar mucho mejor las dinámicas de inestabilidad.

![spearman_completo.png](./img/spearman_completo.png)

## 📊 VISUALIZACIÓN DE DATOS I: DIAGRAMAS DE DISPERSIÓN

Para entender la interacción física entre el mar y la atmósfera, tracé diagramas de dispersión cruzando la temperatura del mar con el resto de variables clave. Tras filtrar las más informativas, seleccioné cuatro relaciones fundamentales para el estudio:
1. **Temperatura del Mar vs. Temperatura del Aire** (evidenciando el acoplamiento térmico).
2. **Temperatura del Mar vs. Presión Máxima** (mostrando la relación inversa y la mayor dispersión en los meses fríos).
3. **Temperatura del Mar vs. Días de Tormenta** (la prueba termodinámica de la inestabilidad).
4. **Temperatura del Mar vs. Temperatura/Viento** (analizando la dinámica de la capa límite).

![dispersion.jpg](./img/dispersion.jpg)

## 📉 VISUALIZACIÓN DE DATOS II: EVOLUCIÓN DECENAL (2x2)

Para observar la tendencia a largo plazo de forma limpia y simétrica, agrupé los datos por décadas acotando el periodo de agosto a diciembre. 

El resultado final se estructuró en una cuadrícula simétrica de **2x2** perfecta para la exposición:
1. Temperatura del Mar
2. Temperatura del Aire
3. Precipitación Extrema
4. Humedad Relativa

(La gráfica de la Velocidad del Viento finalmente fue descartada porque las diferencias no era significativas)

![graficas_decadas.jpg](./img/graficas_decadas.jpg)

## 📉 VISUALIZACIÓN DE DATOS III: TENDENCIAS Y MEDIAS MÓVILES

Finalmente, como última capa de análisis visual, incorporé gráficos con medias móviles (de 10 años) sobre la serie anual completa. Esto me permitió suavizar la alta variabilidad interanual y poner de manifiesto tendencias claras, como el incremento continuo de las temperaturas marina y atmosférica, el descenso de la humedad relativa y un interesante repunte en la velocidad media del viento en la última década.

![media_movil1.jpg](./img/media_movil1.jpg)

## 📉 VISUALIZACIÓN DE DATOS IIII: MAPAS DE CALOR (AÑADIDO DE ÚLTIMA HORA)

Con el propósito de demostrar cómo el verano comienza cada vez antes y las estaciones se vuelven más difusas, inicialmente probé con gráficos ridgeline, aunque el nivel de detalle no era el óptimo. Por ello, decidí recurrir a los mapas de calor: tras evaluar una representación diaria que generaba demasiado ruido visual, opté por estructurar los datos en una matriz quinquenal agrupada por meses, logrando así una visualización mucho más limpia y contundente.

![mapa_calor_dias.png](./img/mapa_calor_dias.png)
![mapa_calor_mensual.png](./img/mapa_calor_mensual.png)
![MapaDeCalorQuinquenal.png](./img/MapaDeCalorQuinquenal.png)